<a href="https://colab.research.google.com/github/Vivekshrotriya1/18_April_Capstone_project/blob/main/18_April_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install azure-ai-documentintelligence

from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
from google.colab import userdata

# Extract Data from Documents
Use Azure Document Intelligence (Form Recognizer)

Steps:
Go to Azure Portal → Create Document Intelligence resource

Use Prebuilt Model → General Document / Forms

Upload sample claim form

 Expected Output:
Extract:

Name

Claim Amount

Date

Policy Number



In [36]:


from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

ENDPOINT = "https://viv54321.cognitiveservices.azure.com/"
KEY = userdata.get('task1')

def analyze_insurance_claim(file_path):
    client = DocumentIntelligenceClient(
        endpoint=ENDPOINT,
        credential=AzureKeyCredential(KEY)
    )

    with open(file_path, "rb") as f:
        file_content = f.read()

    poller = client.begin_analyze_document(
        model_id="prebuilt-layout",
        body=AnalyzeDocumentRequest(bytes_source=file_content),
        features=["keyValuePairs"]
    )

    result = poller.result()

    extracted_data = {}

    if result.key_value_pairs:
        for pair in result.key_value_pairs:
            key = pair.key.content if pair.key else "Unknown"
            value = pair.value.content if pair.value else "N/A"
            print(f"{key} : {value}")
            extracted_data[key] = value

    return extracted_data

result = analyze_insurance_claim(file_name)



important_fields = ["Customer_ID", "Claim_Amount", "Claim_Type", "Location", "Previous_Claims"]

filtered = {}

for k, v in result.items():

    k_cleaned = k.strip().rstrip(":").replace(" ", "_").lower()

    for field in important_fields:
        field_cleaned = field.lower()

        if field_cleaned in k_cleaned or k_cleaned in field_cleaned:
            filtered[field] = v
            break

print(filtered)
print(filtered.keys())

Claim ID: : CLM1001
Date: : 15/01/2025
Name: : Rahul Sharma
Policy Number: : POL123456
Customer ID: : C001
Phone Number: : 9876543210
Email: : rahul.sharma@email.com
Address: : 123, Green Park, Delhi - 110016
Claim Type: : Medical
Location: : Delhi
Claim Amount: : 75000
Description of Incident:
Hospitalization due to viral : fever and related complications.
Date of Incident: : 10/01/2025
Number of Previous Claims: : 2
Signature: : N/A
{'Customer_ID': 'C001', 'Claim_Type': 'Medical', 'Location': 'Delhi', 'Claim_Amount': '75000', 'Previous_Claims': '2'}
dict_keys(['Customer_ID', 'Claim_Type', 'Location', 'Claim_Amount', 'Previous_Claims'])


#Deploy Model as Endpoint
Steps:
Register Model

Create:

Inference script (score.py)

Environment

Deploy → Real-time endpoint



In [42]:
import urllib.request
import json

data = {"Inputs": {"input1": [filtered]}, "GlobalParameters": {}}

body = str.encode(json.dumps(data))

url = 'http://dd50616c-636e-468a-b359-54e18c6c4d9e.koreacentral.azurecontainer.io/score'

api_key = userdata.get('model')
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")


headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))


    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'{"Results": {"WebServiceOutput0": [{"Customer_ID": "C001", "Claim_Amount": 75000, "Claim_Type": "Medical", "Location": "Delhi", "Previous_Claims": 2, "Scored Labels": false, "Scored Probabilities": 0.2886965166571553}]}}'


In [45]:
import requests
import json
from google.colab import userdata

SEARCH_ENDPOINT = "https://task2.search.windows.net"
INDEX_NAME      = "insurance-claims-index"
ADMIN_KEY       = userdata.get('search')
API_VERSION     = "2021-04-30-Preview"

headers_search  = {
    "Content-Type": "application/json",
    "api-key": ADMIN_KEY
}

model_output     = json.loads(result.decode("utf-8"))
fraud_prediction = str(model_output.get("Results", {})
                                   .get("output1", [["Unknown"]])[0][0])
print(f" Fraud Prediction: {fraud_prediction}")


def create_index():
    url           = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}?api-version={API_VERSION}"
    index_schema  = {
        "name": INDEX_NAME,
        "fields": [
            {"name": "id",               "type": "Edm.String", "key": True,  "searchable": False},
            {"name": "Customer_ID",      "type": "Edm.String", "searchable": True,  "filterable": True},
            {"name": "Claim_Amount",     "type": "Edm.Double", "searchable": False, "filterable": True, "sortable": True},
            {"name": "Claim_Type",       "type": "Edm.String", "searchable": True,  "filterable": True},
            {"name": "Location",         "type": "Edm.String", "searchable": True,  "filterable": True},
            {"name": "Previous_Claims",  "type": "Edm.Int32",  "searchable": False, "filterable": True},
            {"name": "Fraud_Prediction", "type": "Edm.String", "searchable": True,  "filterable": True},
        ]
    }
    resp = requests.put(url, headers=headers_search, json=index_schema)


create_index()


def upload_document():
    url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version={API_VERSION}"

    try:
        claim_amt = float(str(filtered.get("Claim_Amount", "0")).replace(",", "").strip())
    except ValueError:
        claim_amt = 0.0

    try:
        prev_claims = int(filtered.get("Previous_Claims", 0))
    except ValueError:
        prev_claims = 0

    document = {
        "value": [{
            "@search.action":   "mergeOrUpload",
            "id":               filtered.get("Customer_ID", "unknown").replace(" ", "_"),
            "Customer_ID":      filtered.get("Customer_ID",     "N/A"),
            "Claim_Amount":     claim_amt,
            "Claim_Type":       filtered.get("Claim_Type",      "N/A"),
            "Location":         filtered.get("Location",        "N/A").strip().rstrip(":"),
            "Previous_Claims":  prev_claims,
            "Fraud_Prediction": fraud_prediction,
        }]
    }

    resp = requests.post(url, headers=headers_search, json=document)


upload_document()


def search(label, query_text="*", filters=None, semantic=False):
    url  = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={API_VERSION}"
    body = {"search": query_text, "top": 10}

    if filters:
        body["filter"] = filters
    if semantic:
        body["queryType"] = "semantic"

    resp    = requests.post(url, headers=headers_search, json=body)
    results = resp.json()

    print(f"\n🔍 {label}")
    print(f"   Filter : {filters or 'None'}")
    for doc in results.get("value", []):
        print(f"   → ID: {doc.get('Customer_ID'):<12} | "
              f"Amount: ₹{doc.get('Claim_Amount'):<10} | "
              f"Type: {doc.get('Claim_Type'):<15} | "
              f"Location: {doc.get('Location'):<12} | "
              f"Fraud: {doc.get('Fraud_Prediction')}")
    return results


r1 = search("Fraud Claim (Semantic)",    query_text="fraud claim",  semantic=True)
r2 = search("Claims above ₹50,000",     filters="Claim_Amount gt 50000")
r3 = search("Fraud Cases in Delhi",     filters="Location eq 'Delhi' and Claim_Type eq 'Fraud'")
r4 = search("All Predicted Fraud",      filters="Fraud_Prediction eq 'Fraud'")

 Fraud Prediction: Unknown

🔍 Fraud Claim (Semantic)
   Filter : None

🔍 Claims above ₹50,000
   Filter : Claim_Amount gt 50000
   → ID: C001         | Amount: ₹75000.0    | Type: Medical         | Location: Delhi        | Fraud: Unknown

🔍 Fraud Cases in Delhi
   Filter : Location eq 'Delhi' and Claim_Type eq 'Fraud'

🔍 All Predicted Fraud
   Filter : Fraud_Prediction eq 'Fraud'
